# Trabajo Práctico — Series Temporales

## Análisis y Pronóstico de la Demanda Eléctrica del AMBA mediante Modelos SARIMA y VAR

---

**Notebook 01 — Inicialización del proyecto y descarga de datos**

| | |
|---|---|
| **Materia** | Series Temporales |
| **Programa** | Maestría en Econometría |
| **Autores** | _(completar)_ |
| **Fecha de creación de este notebook** | 2026-07-23 |

---

## Objetivo de este notebook

Este notebook **no realiza análisis estadístico**. Su única función es dejar preparado el proyecto para los notebooks siguientes:

1. Crear automáticamente toda la estructura de carpetas del proyecto.
2. Registrar información del entorno de ejecución (sistema operativo, versión de Python, memoria disponible).
3. Configurar un sistema de logging que deje constancia en disco de todo lo que se descargó y cuándo.
4. Descargar (o, cuando no sea posible de forma automática, documentar cómo obtener manualmente) las tres series oficiales:
   - Demanda eléctrica del AMBA/SADI — CAMMESA
   - Generación eléctrica del SADI — CAMMESA
   - Temperatura horaria — Servicio Meteorológico Nacional (SMN)
5. Validar lo descargado y generar un resumen de calidad de los datos (cantidad de registros, primera y última fecha, faltantes, frecuencia).

Los notebooks siguientes (`02_limpieza.ipynb` en adelante) parten de lo que este notebook deja en `data/raw/` — nunca vuelven a tocar la etapa de adquisición de datos.

## Organización del proyecto

El proyecto sigue una metodología reproducible de ciencia de datos, con tres niveles de almacenamiento bien diferenciados:

- **`data/raw/`** — Datos exactamente como los entregó la fuente oficial. **Nunca se modifican.** Si en algún momento se sospecha un error de limpieza, siempre se puede volver a este punto de partida.
- **`data/interim/`** — Datos luego de una primera limpieza (parseo, tipado, filtrado por estación/región, eliminación de duplicados). Paso intermedio, todavía no listo para modelar.
- **`data/processed/`** — Series finales, alineadas en un mismo índice horario, listas para el modelado SARIMA/VAR.

Este notebook (`01_descarga_datos.ipynb`) sólo puebla `data/raw/`. Las carpetas `data/interim/` y `data/processed/` se crean vacías, a la espera de `02_limpieza.ipynb`.

**Nota importante sobre las fuentes:** antes de escribir código de descarga se verificaron los endpoints reales de CAMMESA y del SMN — no se asumió que un sitio "publica un CSV histórico" sólo porque lo mencione. Esa verificación llevó a un diseño mixto (automático + manual documentado) que se explica en la sección de CAMMESA más abajo. Se prefiere un pipeline honesto sobre sus límites a uno que prometa una reproducibilidad que la fuente de datos no permite.

In [ ]:
import sys
from pathlib import Path
from datetime import date, timedelta

# Permite ejecutar este notebook tanto desde notebooks/ (caso normal de
# Jupyter, cuyo directorio de trabajo es el del notebook) como desde la raíz
# del proyecto.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils import crear_estructura_proyecto, info_sistema, configurar_logging
from descarga import (
    descargar_smn_rango,
    parsear_archivo_smn,
    descargar_cammesa_demanda_reciente,
    inventariar_cammesa_manual,
    CAMMESA_REGIONES,
    CAMMESA_URL_SINTESIS_MENSUAL,
)

print(f"Directorio raíz del proyecto detectado: {PROJECT_ROOT}")

## 1. Estructura del proyecto

Se crea (si no existe todavía) la totalidad de las carpetas descriptas en el README: `data/`, `notebooks/`, `src/`, `outputs/`, `docs/` y `logs/`, cada una con sus subcarpetas. La función es idempotente: si se vuelve a ejecutar el notebook, no se pisa ni se duplica nada, sólo se informa qué ya existía.

In [ ]:
creadas, existentes = crear_estructura_proyecto(PROJECT_ROOT)

print(f"Carpetas creadas ahora ({len(creadas)}):")
for c in creadas:
    print(f"  + {c}")

print(f"\nCarpetas que ya existían ({len(existentes)}):")
for e in existentes:
    print(f"  = {e}")

## 2. Información del entorno de ejecución

Queda registrado bajo qué condiciones se generaron los datos crudos: sistema operativo, versión de Python y memoria disponible. Es útil para reproducir el entorno en otra máquina y para diagnosticar problemas de rendimiento si la descarga completa del histórico del SMN (~8 años, un pedido HTTP por día) resulta lenta.

In [ ]:
info = info_sistema(PROJECT_ROOT)

print("Proyecto inicializado — información del entorno")
print("=" * 55)
for clave, valor in info.items():
    print(f"{clave:.<35}{valor}")

## 3. Configuración de logging

Todo lo que este notebook descargue queda registrado en `logs/descarga_datos.log`, además de imprimirse en pantalla. Si la descarga completa del SMN se corta a mitad de camino (son miles de pedidos HTTP), queda un registro de hasta dónde se llegó sin depender de haber dejado la celda de Jupyter abierta.

In [ ]:
logger, log_path = configurar_logging(PROJECT_ROOT / "logs")
logger.info("Notebook 01_descarga_datos iniciado")
print(f"Log de esta ejecución: {log_path}")

## 4. Fuente: CAMMESA — demanda y generación eléctrica

**¿Qué es CAMMESA?** La Compañía Administradora del Mercado Mayorista Eléctrico S.A. opera el Sistema Argentino de Interconexión (SADI) y publica sus estadísticas operativas de forma pública.

**¿Por qué se eligió?** Es la única fuente oficial de demanda y generación eléctrica del país, con series a granularidad fina y una justificación teórica clara: la generación responde a la demanda del sistema, y el AMBA concentra una porción muy relevante de esa demanda.

**¿Qué representa cada serie?**
- *Demanda*: potencia (MW) requerida por el sistema en cada instante.
- *Generación*: potencia (MW) inyectada al sistema para abastecer esa demanda.

CAMMESA no publica una "demanda del AMBA" aislada, sino agregados por región. Se usa la región **GBA** (`id_region=426`, agrega Edenor + Edesur + Edelap) como el mejor proxy disponible del AMBA, y **SADI** (`id_region=1002`) como referencia del sistema completo.

### 4.0 Limitación real detectada (léase antes de ejecutar)

Al verificar el endpoint público de CAMMESA se comprobó que **no existe un histórico horario descargable por script para 2018-2024**:

- La API pública (`api.cammesa.com/demanda-svc`) sólo devuelve datos dentro de una **ventana móvil reciente** (aprox. los últimos 7 meses). Pedir una fecha anterior no da error: da una lista vacía, fácil de confundir con "no hay datos ese día" si no se sabe de antemano.
- Los históricos de `datos.gob.ar` son **mensuales por agente**, no horarios de sistema.
- Las páginas del portal con históricos horarios reales ("Demanda Horaria por Tipo", "Síntesis Mensual") usan botones de descarga en JavaScript sin URL de archivo real (`href="#"`) — no automatizables con `requests` sin un navegador simulado (Selenium/Playwright), lo cual excede el alcance de este TP.

Por eso este notebook hace dos cosas distintas para CAMMESA, que conviene no confundir:

### 4.1 Ventana reciente — automatizada vía API pública

Sirve para (a) demostrar el mecanismo de descarga automatizada y (b) mantener el proyecto actualizado con los meses más recientes una vez cargado el histórico manual (sección 4.2). **No reemplaza** el histórico 2018-2024.

In [ ]:
HOY = date.today()
FECHA_FIN_CAMMESA = HOY - timedelta(days=1)        # ayer: hoy puede estar incompleto
FECHA_INICIO_CAMMESA = HOY - timedelta(days=210)   # ~7 meses: ventana verificada

destino_cammesa_reciente = PROJECT_ROOT / "data/raw/cammesa/demanda_reciente_api"

resumen_cammesa = descargar_cammesa_demanda_reciente(
    fecha_inicio=FECHA_INICIO_CAMMESA,
    fecha_fin=FECHA_FIN_CAMMESA,
    destino_dir=destino_cammesa_reciente,
    id_region=CAMMESA_REGIONES["GBA"],
    logger=logger,
)

print(f"Rango solicitado: {FECHA_INICIO_CAMMESA} a {FECHA_FIN_CAMMESA}")
print(f"Días con datos descargados ahora               : {resumen_cammesa.dias_ok}")
print(f"Días ya existentes en disco                     : {resumen_cammesa.dias_ya_existentes}")
print(f"Días fuera de la ventana de CAMMESA (sin datos) : {resumen_cammesa.dias_sin_datos}")
print(f"Días con error de descarga                      : {resumen_cammesa.dias_con_error}")
if resumen_cammesa.dias_sin_datos > 0:
    print(
        "\nNota: los días 'fuera de ventana' son esperables si el rango pedido "
        "excede lo que la API de CAMMESA mantiene disponible. No indica un error."
    )

### 4.2 Histórico horario real — adquisición manual (COMPLETADO)

CAMMESA sí expone el histórico horario real de demanda (por provincia) y generación (por fuente/tecnología), pero como reporte autogenerado desde **Operaciones → Reportes Actuales e Históricos** del portal — no desde "Síntesis Mensual" como se había estimado inicialmente — con un botón de descarga que en ese reporte sí genera un archivo real (a diferencia de los botones JS sin URL encontrados en otras páginas del sitio).

Pasos ya realizados para este proyecto:

1. Ingresar a CAMMESA → Operaciones → Reportes Actuales e Históricos.
2. Generar y descargar el reporte **"Demanda Horaria por Provincia"** y el reporte **"Oferta Total Horaria"** para el rango de fechas deseado.
3. Guardar ambos archivos `.xlsx`, sin modificarlos, en `data/raw/cammesa/sintesis_mensual_manual/`.

**Estructura real verificada de ambos archivos** (una sola hoja cada uno, encabezado en la fila 4, datos desde la fila 5):

| Archivo | Filas de datos | Columnas de valor | Contenido |
|---|---|---|---|
| `Demanda Horaria por Provincia 01012023_30062026.xlsx` | 30.648 | 24 provincias + `TOTAL` | demanda en MWh por hora y por provincia |
| `Oferta Total Horaria 01012023_30062026.xlsx` | 30.648 | 12 fuentes/tecnologías + `TOTAL [MWh]` | generación en MWh por hora, agrupada en Nuclear / Renovable / Térmica / Importación |

Ambos archivos coinciden exactamente en el rango de fechas *realmente* contenido (verificado leyendo la primera y última fila de datos, no sólo el nombre del archivo): **2023-01-01 00h a 2026-06-30 24h** (30.648 horas). Los nombres de archivo originales indicaban un rango levemente distinto para cada uno (`23012023_26062026` y `01012023_01062026`, herencia de cómo se generó cada reporte); se renombraron ambos a `01012023_30062026` para que el nombre refleje el período real y común verificado, en vez de dejar dos fechas de archivo inconsistentes con el contenido. La columna `HORA` usa la convención 1-24 de CAMMESA (paso 1 = 00:00–01:00 … paso 24 = 23:00–00:00); `02_limpieza.ipynb` deberá restar 1 antes de sumarla a `FECHA` para obtener un `datetime` estándar.

Esto **acota el período utilizable del proyecto** a 2023-01-01 – 2026-06-30 (no 2018-2023 como estimaba el README original): aunque la temperatura del SMN sí llega hasta 2018, no tiene sentido cruzar variables con rangos de fechas distintos. Con este rango, la tabla combinada (`series_amba.csv`) tendrá aproximadamente **30.648 registros horarios** — de sobra para todo lo pedido en el TP inicial (ADF/KPSS/SARIMA/VAR/Granger), aunque por debajo de la estimación original de 50.000-60.000 (el README ya fue corregido).

Para la demanda "del AMBA" se usará la columna **`BUENOS AIRES`** como proxy (con la misma salvedad ya documentada en la sección 4: es la demanda de toda la provincia de Buenos Aires, no sólo del conurbano/CABA). Para generación total se usará la columna **`TOTAL [MWh]`**.

**Nota:** en la misma carpeta también quedaron `demanda-historica.csv` / `generacion-historica.csv` (de datos.gob.ar, mensuales por agente) y `BASE_INFORME_MENSUAL_2026-06.zip` (una prueba de la vía "Síntesis Mensual"). Ninguno de los tres hace falta para el pipeline: `02_limpieza.ipynb` sólo va a leer los dos `.xlsx` por nombre. Se pueden dejar sin problema, o moverlos fuera de `data/raw/` si se prefiere la carpeta ordenada.

In [ ]:
directorio_manual = PROJECT_ROOT / "data/raw/cammesa/sintesis_mensual_manual"
inventario_manual = inventariar_cammesa_manual(directorio_manual)

ARCHIVOS_ESPERADOS = ["demanda horaria por provincia", "oferta total horaria"]
encontrados = {clave: None for clave in ARCHIVOS_ESPERADOS}
otros = []

for item in inventario_manual:
    nombre_normalizado = item["archivo"].lower()
    clave = next((c for c in ARCHIVOS_ESPERADOS if c in nombre_normalizado), None)
    if clave:
        encontrados[clave] = item
    elif not item["archivo"].startswith("~$"):  # ignora archivos de bloqueo temporal de Excel
        otros.append(item)

print("Archivos horarios esperados (histórico real de CAMMESA):")
for clave, item in encontrados.items():
    if item:
        print(f"  [OK]    {item['archivo']}  ({item['tamano_kb']} KB)")
    else:
        print(f"  [FALTA] ningún archivo coincide con '{clave}'")
        print(f"          Descargar desde CAMMESA (Operaciones > Reportes Actuales e Históricos).")

if otros:
    print(f"\nOtros archivos presentes (no usados por 02_limpieza.ipynb):")
    for item in otros:
        print(f"  - {item['archivo']}  ({item['tamano_kb']} KB)")

## 5. Fuente: SMN — temperatura horaria (estación Aeroparque)

**¿Qué es el SMN?** El Servicio Meteorológico Nacional es el organismo oficial de meteorología de Argentina. Publica observaciones horarias de todas sus estaciones en archivos de texto de acceso público, un archivo por día.

**¿Por qué la estación Aeroparque?** Es la estación de superficie más representativa del AMBA (CABA), por su ubicación y por tener una serie continua y de buena calidad. Ezeiza es la alternativa habitual para validar o completar faltantes.

**¿Qué representa la serie?** Temperatura de bulbo seco en superficie (°C), medida en punto horario.

A diferencia de CAMMESA, esta fuente **sí es 100% automatizable**: se verificó que el patrón `datohorarioYYYYMMDD.txt` responde de forma confiable desde aproximadamente **enero de 2018** hasta el día anterior a hoy — es decir, el SMN por sí solo alcanzaría para mucho más rango del que en la práctica se puede usar.

**El rango a descargar lo determina CAMMESA, no el SMN.** Como se explicó en la sección 4.2, el histórico horario real de demanda/generación sólo cubre **2023-01-01 a 2026-06-30**. No tiene sentido descargar (ni cruzar) temperatura fuera de esas fechas, así que la descarga de SMN se acota exactamente a ese mismo período para que las tres series terminen con la misma cantidad de registros.

**Sobre los errores de descarga:** con ~1.277 pedidos HTTP (uno por día), es normal que algunos fallen por timeouts o cortes transitorios del servidor del SMN — más aún si el mismo notebook se ejecuta al mismo tiempo desde dos lugares (por ejemplo, esta celda y otra ejecución en paralelo de VS Code), lo que duplica la carga sobre el servidor para las mismas fechas. Por eso cada día que falla se reintenta automáticamente varias veces antes de darlo por perdido (ver la celda siguiente); si igual queda algún día en la lista de errores, alcanza con volver a ejecutar la celda — es resumible, sólo pedirá lo que todavía falte.

In [ ]:
# El rango real del proyecto queda acotado por CAMMESA (sección 4.2), no por
# el SMN (que por sí solo llegaría bastante más atrás, hasta ~2018).
FECHA_INICIO_SMN = date(2023, 1, 1)
FECHA_FIN_SMN = date(2026, 6, 30)

# MODO_PRUEBA=True descarga sólo los últimos 30 días de ese rango, para
# verificar en segundos que el mecanismo funciona antes de lanzar la
# descarga completa (~1.277 días, un pedido HTTP por día: del orden de
# 20-30 minutos). Dejar en False para la descarga real del proyecto.
MODO_PRUEBA = False

if MODO_PRUEBA:
    FECHA_INICIO_SMN = FECHA_FIN_SMN - timedelta(days=30)

destino_smn = PROJECT_ROOT / "data/raw/smn"

print(f"Modo prueba: {MODO_PRUEBA}")
print(
    f"Rango a descargar: {FECHA_INICIO_SMN} a {FECHA_FIN_SMN} "
    f"({(FECHA_FIN_SMN - FECHA_INICIO_SMN).days + 1} días)"
)

In [ ]:
# Cada día que falle por un error de red (timeout, conexión rechazada, etc.)
# se reintenta automáticamente hasta REINTENTOS_SMN veces, esperando
# ESPERA_REINTENTO_SMN_SEG entre intentos, antes de contarlo como error
# definitivo. Un día "sin datos" (el SMN responde que el archivo no existe)
# NO se reintenta: no es un problema transitorio, es una respuesta válida.
REINTENTOS_SMN = 3
ESPERA_REINTENTO_SMN_SEG = 5.0

resumen_smn = descargar_smn_rango(
    fecha_inicio=FECHA_INICIO_SMN,
    fecha_fin=FECHA_FIN_SMN,
    destino_dir=destino_smn,
    logger=logger,
    pausa_seg=0.3,
    reintentos=REINTENTOS_SMN,
    espera_reintento_seg=ESPERA_REINTENTO_SMN_SEG,
)

print(f"Días descargados ahora     : {resumen_smn.dias_ok}")
print(f"Días ya existentes en disco: {resumen_smn.dias_ya_existentes}")
print(f"Días sin datos en el SMN   : {resumen_smn.dias_sin_datos}")
print(f"Días con error de descarga : {resumen_smn.dias_con_error} (ya reintentados {REINTENTOS_SMN} veces cada uno)")
if resumen_smn.errores:
    print(f"Fechas que fallaron incluso después de los reintentos (volver a ejecutar esta celda más tarde): {resumen_smn.errores}")

## 6. Parseo y verificación de la serie de temperatura

Se parsean únicamente los archivos ya descargados en `data/raw/smn/`, filtrando por la estación Aeroparque, para poder calcular el resumen de calidad de la sección siguiente. **Este paso no modifica los archivos crudos**: la limpieza y el guardado en `data/interim/` / `data/processed/` es responsabilidad de `02_limpieza.ipynb`.

In [ ]:
registros_temperatura = []
for archivo in sorted(destino_smn.glob("datohorario*.txt")):
    registros_temperatura.extend(parsear_archivo_smn(archivo))

print(f"Registros horarios de temperatura parseados: {len(registros_temperatura)}")

## 7. Resumen de calidad de los datos descargados

Antes de pasar a `02_limpieza.ipynb`, se deja constancia de: cantidad de registros, primera y última fecha disponible, cantidad de horas faltantes respecto de una serie horaria completa, y si la frecuencia observada es efectivamente horaria.

In [ ]:
def resumen_serie_horaria(fechas_horas, nombre_serie):
    """fechas_horas: lista de tuplas (date, hora) ya parseadas."""
    if not fechas_horas:
        print(f"[{nombre_serie}] No hay datos descargados todavía.")
        return

    momentos = sorted(set(fechas_horas))
    primero, ultimo = momentos[0], momentos[-1]
    horas_esperadas = ((ultimo[0] - primero[0]).days + 1) * 24
    faltantes = horas_esperadas - len(momentos)

    print(f"[{nombre_serie}]")
    print(f"  Registros                 : {len(momentos)}")
    print(f"  Primer momento             : {primero[0]} {primero[1]:02d}:00")
    print(f"  Último momento             : {ultimo[0]} {ultimo[1]:02d}:00")
    print(f"  Horas esperadas en el rango: {horas_esperadas}")
    print(f"  Horas faltantes (aprox.)   : {faltantes}")
    print()


resumen_serie_horaria(
    [(r["fecha"], r["hora"]) for r in registros_temperatura],
    "Temperatura (SMN - Aeroparque)",
)

In [ ]:
import json

momentos_demanda = []
for archivo in sorted(destino_cammesa_reciente.glob("*.json")):
    datos = json.loads(archivo.read_text(encoding="utf-8"))
    for punto in datos:
        # "fecha" viene con formato tipo "2026-07-20T00:05:00.000-0300"
        ts = punto["fecha"]
        fecha_pt = date.fromisoformat(ts[:10])
        hora_pt = int(ts[11:13])
        momentos_demanda.append((fecha_pt, hora_pt))

resumen_serie_horaria(
    momentos_demanda,
    "Demanda (CAMMESA GBA - ventana reciente, agregada a nivel hora)",
)

archivos_historico_ok = sum(1 for v in encontrados.values() if v)
print(
    f"Archivos del histórico horario real de CAMMESA disponibles: {archivos_historico_ok}/2 "
    "(ver sección 4.2 si falta alguno)"
)

## 8. Cierre

```
Proyecto inicializado correctamente
```

- [x] Estructura de carpetas creada/verificada
- [x] Información del entorno registrada
- [x] Logging configurado (`logs/descarga_datos.log`)
- [x] Temperatura (SMN, Aeroparque) descargada de forma automática
- [x] Demanda reciente (CAMMESA, ventana ~7 meses) descargada de forma automática
- [x] Histórico horario real de demanda (por provincia) y generación (por fuente/tecnología) descargado manualmente desde CAMMESA — rango real verificado: 2023-01-01 a 2026-06-30 (ver sección 4.2)

**Próximo paso:** continuar con `02_limpieza.ipynb`, que tomará los dos `.xlsx` de `data/raw/cammesa/sintesis_mensual_manual/` junto con `data/raw/smn/`, los acotará al período común (2023-01-01 – 2026-06-30), y generará `demanda.csv`, `generacion.csv` y `temperatura.csv` en `data/processed/`, con el mismo índice horario `datetime`.

In [ ]:
logger.info("Notebook 01_descarga_datos finalizado")
print("Proyecto listo para continuar con 02_limpieza.ipynb")